# Exploratory Data Analysis

1. [Comparing Sectors](#Comparing-Sectors)

## Comparing Sectors

In [71]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px


In [65]:
# data from https://www.climatewatchdata.org/data-explorer/ndc-content?ndc-content-categories=sectoral_information&ndc-content-indicators=All%20Selected&ndc-content-locations=All%20Selected&ndc-content-sectors=All%20Selected&page=1
# https://www.climatewatchdata.org/data-explorer/historical-emissions?historical-emissions-data-sources=climate-watch&historical-emissions-gases=all-ghg&historical-emissions-regions=All%20Selected&historical-emissions-sectors=agriculture%2Cbuilding%2Celectricity-heat%2Cenergy%2Cindustrial-processes%2Cland-use-land-use-change-and-forestry%2Ctransportation%2Cwaste%2Cmanufacturing-construction&page=1&sort_col=sector&sort_dir=ASC

# get Sectors
df = pd.read_csv(os.path.join(sector_data_dir, 'historical_emissions','historical_emissions.csv'))
sectors = df['Sector'].unique()
sectors

array(['Agriculture', 'Building', 'Electricity/Heat', 'Energy',
       'Industrial Processes', 'Land Use, Land-Use Change and Forestry',
       'Manufacturing/Construction', 'Transportation', 'Waste'],
      dtype=object)

In [66]:
# keep only important columns

YEAR = '2023' # latest = 2023

df = df[['Country', 'Sector', 'Gas', 'Unit', YEAR]]
df

,Country,Sector,Gas,Unit,2023
0,Afghanistan,Agriculture,All GHG,MtCO₂e,16.07
1,Angola,Agriculture,All GHG,MtCO₂e,35.60
2,Albania,Agriculture,All GHG,MtCO₂e,2.08
3,Andorra,Agriculture,All GHG,MtCO₂e,0.00
4,United Arab Emirates,Agriculture,All GHG,MtCO₂e,2.26
...,...,...,...,...,...
1747,Samoa,Waste,All GHG,MtCO₂e,0.00
1748,Yemen,Waste,All GHG,MtCO₂e,3.78
1749,South Africa,Waste,All GHG,MtCO₂e,4.24
1750,Zambia,Waste,All GHG,MtCO₂e,6.85


In [69]:
# enusre 'Gas' and 'Unit' are all equal
len(df['Gas'].unique()) == 1 and len(df['Unit'].unique()) == 1

True

In [48]:
# take note of what countries do not have data, should remove these from all other sectors
# and update df
sector_groups = df.groupby('Sector')['Country'].apply(set)

country_set = sector_groups['Agriculture']
# type(country_set)
for sector in sector_groups:
    country_set = country_set.intersection( sector)

print(f"Number of countries with data across all sectors: {len(country_set)}")

df = df[df['Country'].isin(country_set)]


Number of countries with data across all sectors: 182


In [82]:
# get numbers per Sector
em_per_sector = {}
for sector in sectors:
    em_per_sector[sector] = round(float(df[df['Sector'] == sector][YEAR].sum()),2)

# simplify name
em_per_sector['Land-Use/Forestry'] = em_per_sector.pop('Land Use, Land-Use Change and Forestry')

# sort 
em_per_sector = dict(sorted(em_per_sector.items(), key=lambda x: x[1])) # , reverse=True


fig = px.bar(y=em_per_sector.keys(), x=em_per_sector.values(), orientation='h')
fig.update_layout(title='Global Gas Emmisions Per Sector (MtCO2e)',
                  title_x=0.5,
                  xaxis_title="Gas Emissions",
                  yaxis_title="Sector")
fig.show()

# display values
em_per_sector

{'Land-Use/Forestry': 1028.96,
 'Waste': 3997.23,
 'Industrial Processes': 6475.01,
 'Building': 6729.72,
 'Manufacturing/Construction': 12713.08,
 'Agriculture': 12847.62,
 'Transportation': 16488.62,
 'Electricity/Heat': 34693.54,
 'Energy': 78773.02}

In [ ]:
# columns 
cement_co2	Annual CO₂ emissions from cement
co2_per_unit_energy	CO₂ emissions per unit energy	Carbon dioxide (CO₂) emitted per unit of primary energy consumption, measured in grams of CO₂ per kilowatt-hour.	grams per kilowatt-hour (g/kWh)	Global Carbon Budget (2025) [https://globalcarbonbudget.org/]; U.S. Energy Information Administration - International Energy Data (2026) [https://www.eia.gov/opendata/bulkfiles.php]; Energy Institute - Statistical Review of World Energy (2025) [https://www.energyinst.org/statistical-review/]
	coal_co2	Annual CO₂ emissions from coal
    gas_co2	Annual CO₂ emissions from gas

ghg_excluding_lucf_per_capita	Per capita greenhouse gas emissions from fossil fuels and industry
oil_co2	Annual CO₂ emissions from oil


,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2024)
1,year,Year,Year of observation.,NaN,Our World in Data - Regions (2024)
2,iso_code,ISO code,ISO 3166-1 alpha-3 three-letter country codes.,NaN,International Organization for Standardization...
3,population,Population,"Population by country, available from 10,000 B...",people,Population based on various sources (2024) [ht...
4,gdp,Gross domestic product (GDP),Total economic output of a country or region p...,international-$ in 2011 prices ($),Bolt and van Zanden – Maddison Project Databas...
...,...,...,...,...,...
74,temperature_change_from_n2o,Change in global mean surface temperature caus...,Measured in °C.,°C,Jones et al. - National contributions to clima...
75,total_ghg,Annual greenhouse gas emissions including land...,Greenhouse gas emissions are measured in tonne...,million tonnes (Mt),Jones et al. - National contributions to clima...
76,total_ghg_excluding_lucf,Annual greenhouse gas emissions from fossil fu...,Greenhouse gas emissions are measured in tonne...,million tonnes (Mt),Jones et al. - National contributions to clima...
77,trade_co2,Annual CO₂ emissions embedded in trade,Annual net carbon dioxide (CO₂) emissions embe...,million tonnes (Mt),Global Carbon Budget (2025) [https://globalcar...
